# agent_llm cache_key 마이그레이션 — 호출 라벨(service_name/task_name) 키 제거

LLM 응답 캐시(`agent_llm`)의 `cache_key`(=`_id`)는 canonical JSON 의 SHA-256 해시다.
코드 변경으로 **canonical 에서 호출 라벨을 제외**(LLM 출력은 입력에만 의존)하면서,
기존 블락은 라벨이 포함된 옛 키라 그대로 두면 새 코드가 못 맞춘다(전부 cache miss).

## 방식 (Option B — 옛 블락/agent_output 무변경)
- 옛 블락은 그대로 둔다 → agent_output(RM/ESG) 등에 저장된 옛 cache_key 링크 보존
- 라벨 제거한 **새 _id 블락을 원본에 추가** → 앞으로 같은 입력은 새 키로 여기 HIT
- agent_output 은 건드리지 않는다

## 원본 / 백업 DB 분리
- **원본(SOURCE_DB) = 마이그레이션 대상.** apply 시 새 블락이 여기 삽입된다. (운영: `kw_ai`)
- **백업(BACKUP_DB) = 안전 사본 저장 위치.** 원본과 분리. (`kw_ai_dev`)
- 먼저 시험하려면 SOURCE_DB 를 `kw_ai_dev` 로 두고 리허설한다.

## 재실행(incremental) 안내
- apply 는 **새 키 블락이 이미 있으면 skip**, 없는 것만 삽입한다 → 여러 번 돌려도 안전.
- 코드 미배포로 옛 키 블락이 계속 쌓여도, **새 백업 스냅샷으로 다시 돌리면 빠진 것만** 복제된다.
  (재실행 시 BACKUP_COLL 이름을 새로 주거나 이전 백업을 먼저 비워야 한다 — 3번 셀이 빈 백업을 요구)
- 단, 근본은 **코드 배포 먼저** — 배포 후 옛 키 생성이 멈추면 1회 마이그레이션으로 끝난다.

## 절차 (셀 순서대로)
1. 매니저 설치 (로컬 editable)  2. config/연결  3. 백업 복제(실행 즉시, 비파괴)
4. 미리보기(read-only)  5. apply(`CONFIRM_APPLY=True`)  6. 검증  7. 백업 삭제(`CONFIRM_DELETE_BACKUP=True`)

> ⚠️ apply / 백업삭제만 플래그로 가드된다. 백업(3)은 실행하면 바로 복제된다.
> 라벨이 다른 동일 입력은 하나의 새 키로 합쳐진다(collapse) — 의도된 동작.

In [ ]:
# 0. 매니저 로컬 설치 (GitHub URL 아님 — 로컬 clone 최신본을 editable 로)
#    노트북 CWD = Kiwoom-Manager/llm-manager/migrations 기준 상대경로
%pip install -q \
    -e ../../mdb-manager \
    -e ../../redis-manager \
    -e ../../file-manager \
    -e ..

In [ ]:
import json

from mdb_manager import MongoDBManager
from llm_manager._internal.keying import canonicalize_llm_input, make_cache_key

# ── DB 설정 (원본과 백업을 분리) ──────────────────────────
SOURCE_DB   = "kw_ai"      # 원본 = 마이그레이션 대상(운영). 리허설은 "kw_ai_dev"
BACKUP_DB   = "kw_ai_dev"  # 백업 저장 위치 (원본과 분리)
COLL        = "agent_llm"
BACKUP_COLL = "agent_llm_backup_20260616"
CALLER      = "llm-manager"   # agent_llm write ACL 용 caller

mdb = MongoDBManager().with_caller(CALLER)
col    = mdb.get_database(SOURCE_DB, COLL)           # 원본 (읽기 + apply 시 새블락 삽입)
backup = mdb.get_database(BACKUP_DB, BACKUP_COLL)    # 백업


def fetch_all(coll, batch=1000):
    """find 가 서버 기본 limit 에 잘리지 않도록 페이지네이션으로 전체 조회."""
    out, skip = [], 0
    while True:
        page = coll.find({}, limit=batch, skip=skip)
        if not page:
            break
        out.extend(page)
        if len(page) < batch:
            break
        skip += batch
    return out


print(f"원본:  {SOURCE_DB}.{COLL}  (count={col.count()})")
print(f"백업:  {BACKUP_DB}.{BACKUP_COLL}  (count={backup.count()})")

In [ ]:
# ① 백업 복제 (원본 → 백업DB). 비파괴 — 이 셀 실행 시 즉시 복제된다.
# 재실행하려면 BACKUP_COLL 이름을 새로 주거나, 이전 백업을 비운 뒤 진행한다.
assert backup.count() == 0, f"백업 {BACKUP_DB}.{BACKUP_COLL} 이 비어있지 않음 — 이름/DB 확인 후 재시도"

docs = fetch_all(col)
print("복제 대상:", len(docs))

BATCH = 500
for i in range(0, len(docs), BATCH):
    backup.insert_many(docs[i:i + BATCH])   # _id 보존

print("백업 완료:", backup.count(), "/ 원본:", col.count())
assert backup.count() == len(docs), "백업 수 불일치 — 중단하고 확인"

In [ ]:
def new_key(doc):
    """옛 canonical_json 에서 라벨 키 제거 → 새 canonical / 새 _id 도출.

    저장된 canonical_json 만으로 계산 가능(원본 llm_input 불필요). 실제 keying 함수를
    재사용하므로 런타임이 만드는 새 키와 정확히 일치한다(검증 완료).
    """
    sub = json.loads(doc["canonical_json"])
    sub.pop("service_name", None)
    sub.pop("task_name", None)
    new_canonical = canonicalize_llm_input(sub)
    return make_cache_key(new_canonical), new_canonical


# ② 미리보기 (read-only) — 백업 스냅샷 기준
src = fetch_all(backup)
new_to_old, collisions, unchanged = {}, 0, 0
for d in src:
    nid, _ = new_key(d)
    if nid == d["_id"]:
        unchanged += 1
    if nid in new_to_old:
        collisions += 1
    new_to_old[nid] = d["_id"]

print("총:", len(src))
print("키 그대로(이미 라벨 없음):", unchanged)
print("새 _id 종류:", len(new_to_old))
print("collapse(여러 옛블락 → 같은 새키):", collisions)
print("--- 샘플 ---")
for d in src[:5]:
    nid, _ = new_key(d)
    tag = "(unchanged)" if nid == d["_id"] else ""
    print(f"  {d['_id'][:12]}… → {nid[:12]}… {tag}")

In [ ]:
# ③ apply — 원본(SOURCE_DB)에 새 블락 삽입 (incremental: 이미 있으면 skip)
CONFIRM_APPLY = False   # 실제 삽입하려면 True 로 바꾸고 재실행

if not CONFIRM_APPLY:
    print("미적용 — 미리보기 확인 후 CONFIRM_APPLY=True 로 재실행")
else:
    src = fetch_all(backup)                 # 고정 스냅샷 기준
    inserted = already = skipped = 0
    for d in src:
        nid, ncanon = new_key(d)
        if nid == d["_id"]:
            skipped += 1                     # 이미 라벨 없는 키 → 추가 불필요
            continue
        if col.find_one({"_id": nid}) is not None:
            already += 1                     # 이미 복제됨 → 건너뜀 (재실행 안전)
            continue
        new_doc = {**d, "_id": nid, "canonical_json": ncanon}
        col.replace_one({"_id": nid}, new_doc, upsert=True)
        inserted += 1
    print("삽입:", inserted, "| 이미존재 skip:", already, "| 키동일 skip:", skipped)
    print(f"원본 {SOURCE_DB}.{COLL} 현재 수:", col.count())

In [ ]:
# ④ 검증 — 백업의 모든 블락이 새 키로 원본(SOURCE_DB)에 존재하는가
src = fetch_all(backup)
missing, empty_result = [], []
for d in src:
    nid, _ = new_key(d)
    got = col.find_one({"_id": nid})
    if got is None:
        missing.append(d["_id"])
    elif got.get("result") in (None, {}):
        empty_result.append(nid)

print("검증 대상:", len(src))
print("누락(새 키 블락 없음):", len(missing))
print("result 비어있음:", len(empty_result))
print("✅ OK" if not missing and not empty_result else "❌ 확인 필요 — 백업 삭제 금지")
if missing[:5]:
    print("누락 샘플:", missing[:5])

In [ ]:
# ⑤ 백업 삭제 — 검증 ✅ OK 확인 후에만
CONFIRM_DELETE_BACKUP = False   # 검증 OK 확인 후 True 로

if not CONFIRM_DELETE_BACKUP:
    print("백업 유지 — 검증 OK 확인 후 CONFIRM_DELETE_BACKUP=True 로 재실행")
else:
    backup.drop()
    print(f"백업 삭제 완료: {BACKUP_DB}.{BACKUP_COLL}")